# CIS I — Assignment 4: Robot Virtual Fixtures

A robot holds a surgical tool. You will write its **midlevel controller**. Every
$\Delta t = 20$ ms the controller reads the robot state, solves a small constrained least-squares
problem for the joint increment $\Delta\vec q$, and commands $\dot{\vec q} = \Delta\vec q/\Delta t$.
It runs in two phases:

1. **Approach**: move autonomously from $F^{init}_{RT}$ to $F^{start}_{RT} = F_{tube}$.
   The tip moves in a straight line, the tool rotates about a fixed axis, and both are coordinated.
2. **Cooperative control**: a (simulated) surgeon's hand pushes the tool down a 10 mm-radius tube.
   Admittance gains $\kappa_S$ and SDF virtual fixtures keep the tip away from the critical anatomy,
   keep it under $v^{safe}_S$ near each structure, and stop it within 0.2 mm of the stopper at the bottom.

**Your code lives in `src/student/`** (each file's docstring says what it needs):

| file | you write |
|---|---|
| `jacobian.py` | $J_{RW}(\vec q)$, $J_{RT}(\vec q)$ (body Jacobians, handout convention) |
| `wrench.py` | resolving the wrist force/torque at the tool tip |
| `virtual_fixtures.py` | joint-limit bounds, linearized SDF constraints, centering bias |
| `approach.py` | motion profile, pose interpolation, one optimization step |
| `gains.py` | **your** $\kappa_S$ design |
| `cooperative.py` | admittance law, one optimization step |

The loops `move_tool_to` and `cooperative_control_mode` (bottom of `approach.py` / `cooperative.py`)
are the handout's pseudocode and are provided. So are the robot, anatomy, solver, logging,
evaluation and plotting (`src/simulation`, `src/tools`).

**Conventions and handout clarifications** (also in `src/simulation/constants.py`):
* Wrenches are $\vec\zeta = [\vec\tau;\ \vec\phi]$: torques first.
* $Rot(r_x, r_y, r_z)$ in $F^{init}_{RT}$ is ZYX: $R = R_z(r_z)R_y(r_y)R_x(r_x)$.
* $d^{range} = 10$ mm for every structure.
* Donut distance: the closest point on the core circle is $[a_x, a_y, 0]$, not $[a_x, a_y, x_z]$.
* The hand's force tilts by at most ~20° from vertical while its direction spins slowly.
* At $F^{start}_{RT}$, joint 2 is **exactly on its +300 mm travel limit**. The tip cannot move toward
  $+y$ of the robot inside the tube, so your controller must respect that.

Run `pytest tests/test_student.py -m "not slow"` as you go.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import matplotlib.pyplot as plt

from src import simulation as sim, Frame, Rot, vct3
from src.simulation import constants as C
from src.tools import (TrajectoryLog, evaluate_approach, evaluate_cooperative, print_report,
                       pose_error_body, frame_Rp, tilt_angle)
from src.tools import visualization as viz
from src.student import jacobian, wrench, gains, virtual_fixtures, approach, cooperative

np.set_printoptions(precision=4, suppress=True)

## Step 0: Meet the robot and the anatomy

`create_robot()` returns a stopped robot holding the tool at $F^{init}_{RT}$. `read_state()` and
`send_command(v)` are the handout's `ReadState` and `SendCommand`. `send_command` enforces the joint
velocity, acceleration and travel limits, and returns the `rc` bit mask.

`build_scene()` places the 8 structures S1–S8 of the handout. Each structure has `type`, `v_safe`,
`d_safe`, `d_range`, `distance_at(p_R)` and `gradient_at(p_R)`, with robot-frame positions in and
robot-frame gradients out. Positive distance means the tip is allowed there, and the gradient points
away from the forbidden region. That holds for the tube wall too.

In [ ]:
robot = sim.create_robot()
scene = sim.build_scene()
print(robot)
print("q_init =", robot.read_state().q)
print("q at F_start =", robot.tool_inv_kins(C.F_RT_START), " (q2 limit:", C.Q_MAX[1], ")")
for s in scene:
    print(" ", s)

In [ ]:
viz.plot_scene(scene);
viz.plot_slices(scene);

## Step 1: Jacobians (`jacobian.py`)

The handout's convention is $F(\vec q+\Delta\vec q) = F(\vec q)\,\Delta F$ with
$\Delta F \approx [I + sk(\vec\alpha),\ \vec\epsilon]$ and $[\vec\alpha;\vec\epsilon] = J\,\Delta\vec q$.
The cell below compares your $J_{RT}$ with finite differences.

*If you used an AI system to help, record the prompt(s) and how you checked the result in the last section.*

In [ ]:
q = robot.read_state().q
J = jacobian.jacobian_RT(q, robot.F_WT)
h = 1e-6
J_num = np.column_stack([pose_error_body(robot.tool_pose(q - h * e), robot.tool_pose(q + h * e)) / (2 * h)
                         for e in np.eye(6)])
print("max |J - J_numerical| =", np.abs(J - J_num).max())

## Step 2: Resolve the wrist wrench at the tip (`wrench.py`)

In [ ]:
st = robot.read_state()
zeta_T = wrench.resolve_wrench_to_tool(st.zeta_W, robot.F_WT)
zeta_RT = wrench.wrench_in_robot_frame(zeta_T, robot.tool_pose(st.q).R.matrix)
print("zeta_W  =", st.zeta_W)
print("zeta_T  =", zeta_T)
print("phi_RT  =", zeta_RT[3:], " (should equal the hand force:", robot.force_model.force_R(st.t), ")")

## Step 3: The approach move (`approach.py`, `virtual_fixtures.joint_motion_bounds`)

**Your formulation (edit this cell):** describe the motion profile $s(t)$, how you chose its
parameters so the joint limits hold, and the optimization problem you solve each period
(objective terms, weights, constraints).

> TODO

In [ ]:
robot = sim.create_robot()
log = TrajectoryLog(scene.names)
plan = approach.move_tool_to(robot, C.F_RT_START, log)
print(plan)
print("rc flags raised:", robot.rc_counts)
res_approach = evaluate_approach(log, C.F_RT_INIT, C.F_RT_START, robot.limits, robot.dt)
print_report(res_approach)

In [ ]:
viz.plot_approach(log, C.F_RT_INIT, C.F_RT_START);
viz.plot_joint_usage(log, robot.limits, robot.dt, phase="approach");

## Step 4: Gain design (`gains.py`)

**Your design and rationale (edit this cell; this part is graded on its reasoning):**
* the shape of $v_S(d) = \kappa_S(d)\,\phi^{max}$ for each structure type and why,
* how it enforces $v^{safe}_S$ inside $d^{safe}_S$, with what margin, and why the margin is needed,
* how the acceleration limits shaped your choice of top speed inside the tube,
* how the gains combine when several structures are in range.

> TODO

In [ ]:
viz.plot_speed_limits(gains.speed_limit, scene);

## Step 5: Cooperative control with virtual fixtures (`virtual_fixtures.py`, `cooperative.py`)

**Your formulation (edit this cell):** write down the optimization problem you solve each period.
Cover the objective terms and their weights, the linearized SDF constraints (clearances, and how fast
clearance may be used up), the centering bias, keeping the tool vertical, the joint limits, and what
you do if the problem is infeasible.

> TODO

In [ ]:
d_final = cooperative.cooperative_control_mode(robot, scene, log)
print(f"halted {d_final:.3f} mm from the stopper at t = {robot.t:.2f} s; rc flags: {robot.rc_counts}")
res_coop = evaluate_cooperative(log, scene, robot.limits, robot.dt)
print_report(res_coop)

In [ ]:
viz.plot_scene(scene, log);
viz.plot_slices(scene, log);
viz.plot_distances(log, scene);
viz.plot_speed(log, scene);
viz.plot_joint_usage(log, robot.limits, robot.dt, phase="cooperative");

## Step 6: Program output

The required output is the time trace of the tool pose and the distances to every structure.

In [ ]:
os.makedirs("results", exist_ok=True)
log.to_csv("results/trajectory.csv")
print(len(log), "samples written to results/trajectory.csv")

## Step 7: Robustness

Graders also run your controller with other hand-force profiles. Try a few here: no wobble, a larger
tilt, faster wobble, and a constant sideways push. Comment on anything that fails and why.

In [ ]:
profiles = {
    "no wobble":      sim.HandForceModel(tilt_max=0.0),
    "tilt 0.6 rad":   sim.HandForceModel(tilt_max=0.6),
    "fast wobble":    sim.HandForceModel(omega_x=0.2 * np.pi, omega_z=0.4 * np.pi),
    "push toward +x": sim.HandForceModel(force_fn=lambda t: np.array([1.5, 0.0, -4.7])),
}
for name, hand in profiles.items():
    r, s, L = sim.create_robot(force_model=hand), sim.build_scene(), TrajectoryLog()
    try:
        approach.move_tool_to(r, C.F_RT_START, L)
        cooperative.cooperative_control_mode(r, s, L)
        res = evaluate_cooperative(L, s, r.limits, r.dt)
        print(f"{name:16s} {'PASS' if res['passed'] else 'FAIL'}  {res['duration']:.1f} s",
              [k for k, v in res["checks"].items() if not v])
    except RuntimeError as e:
        print(f"{name:16s} ERROR {e}")

## AI usage

List every prompt you gave an AI system (including the inputs and outputs you asked for), which
functions it helped with, and how you checked the result.

> TODO